In [1]:
!unzip -o archive.zip
import numpy as np
np.set_printoptions(suppress=True)

Archive:  archive.zip
  inflating: insurance.csv           


## Task 1
### Before calculating
The 19,600 average is true mathematically, but it is misleading.
One salary (95,000 for the CEO) pulls the average up. Most employees earn around 7,000.

In [2]:
salaries = np.array([6000, 6500, 7000, 7200, 7500, 8000, 95000])

print("Mean:", salaries.mean())
print("Median:", np.median(salaries))

q1, q3 = np.percentile(salaries, [25, 75])
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Limits:", low, "to", high)
print("Outliers:", salaries[(salaries < low) | (salaries > high)])

no_ceo = salaries[:-1]
print("Without CEO -> Mean:", round(no_ceo.mean(), 1), "| Median:", np.median(no_ceo))

Mean: 19600.0
Median: 7200.0
Q1: 6750.0 | Q3: 7750.0 | IQR: 1000.0
Limits: 5250.0 to 9250.0
Outliers: [95000]
Without CEO -> Mean: 7033.3 | Median: 7100.0


### Sensitivity
Without the CEO, the mean dropped from 19,600 to 7,033, but the median only moved from 7,200 to 7,100.
So the mean is very sensitive to outliers, and the median is not.

### An outlier is not always an error
- Real and important (keep it): a customer who spends 50,000 SAR in one day at a store. It is real, and it is a VIP customer.
- Clear error (fix it): a student's age written as 250 instead of 25.

### z-score in two lines
z-score tells how far a value is from the average, measured in "normal steps".
Like a tall friend: if most people are 170 cm, someone at 200 cm is many steps away, so his z-score is high.

## Task2

In [3]:
data = np.genfromtxt("insurance.csv", delimiter=",", skip_header=1, usecols=(0, 2, 3, 6))
print("shape:", data.shape)
print(data[:3])

age = data[:, 0]
bmi = data[:, 1]
children = data[:, 2]
charges = data[:, 3]

for name, col in [("age", age), ("bmi", bmi), ("children", children), ("charges", charges)]:
    print(f"{name:9} mean={col.mean():10.2f} median={np.median(col):10.2f} std={col.std():10.2f} min={col.min():10.2f} max={col.max():10.2f}")

print("Missing values:", np.isnan(data).sum())
print("No children %:", round((children == 0).mean() * 100, 1))

shape: (1338, 4)
[[   19.        27.9        0.     16884.924 ]
 [   18.        33.77       1.      1725.5523]
 [   28.        33.         3.      4449.462 ]]
age       mean=     39.21 median=     39.00 std=     14.04 min=     18.00 max=     64.00
bmi       mean=     30.66 median=     30.40 std=      6.10 min=     15.96 max=     53.13
children  mean=      1.09 median=      1.00 std=      1.21 min=      0.00 max=      5.00
charges   mean=  13270.42 median=   9382.03 std=  12105.48 min=   1121.87 max=  63770.43
Missing values: 0
No children %: 42.9


### Notes
The column where the mean is far from the median is charges (mean 13,270 vs median 9,382).
This means most customers pay a low amount, but a few pay very high amounts that pull the mean up.
The std of charges is very large (12,105), almost as big as the mean itself, so the charges are very spread out.
The other columns (age, bmi, children) have a mean close to the median, so they are balanced.

In [4]:
def iqr_outliers(col, name):
    q1, q3 = np.percentile(col, [25, 75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (col < low) | (col > high)
    print(f"{name}: Q1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} limits=({low:.2f}, {high:.2f}) outliers={mask.sum()} ({mask.mean()*100:.1f}%)")
    return mask

charges_out = iqr_outliers(charges, "charges")
bmi_out = iqr_outliers(bmi, "bmi")

print("Top 5 charges:", np.sort(charges)[-5:])
print("Top 5 bmi:", np.sort(bmi)[-5:])

for name, col in [("charges", charges), ("bmi", bmi)]:
    z = (col - col.mean()) / col.std()
    print(name, "|z| > 3:", (np.abs(z) > 3).sum())

print("Mean charges with outliers:", round(charges.mean(), 2))
print("Mean charges without outliers:", round(charges[~charges_out].mean(), 2))

charges: Q1=4740.29 Q3=16639.91 IQR=11899.63 limits=(-13109.15, 34489.35) outliers=139 (10.4%)
bmi: Q1=26.30 Q3=34.69 IQR=8.40 limits=(13.70, 47.29) outliers=9 (0.7%)
Top 5 charges: [55135.40209 58571.07448 60021.39897 62592.87309 63770.42801]
Top 5 bmi: [48.07 49.06 50.38 52.58 53.13]
charges |z| > 3: 7
bmi |z| > 3: 4
Mean charges with outliers: 13270.42
Mean charges without outliers: 9927.75


### IQR vs z-score
IQR found about 139 outliers in charges, but z-score (|z| > 3) found only 7.
IQR uses the middle 50% of the data (Q1 and Q3), and charges are skewed, so the upper limit is low (about 34,489).
z-score uses the mean and std, which are already pulled up by the big charges, so only extreme values pass |z| > 3.

### Mean with and without outliers
Mean charges with outliers: 13,270. Without them: 9,928. The outliers raise the mean by about 3,300.

### Decision: are these errors?
About 10% of customers are "outliers". That is too many to be typing mistakes.
When I open the file, there is a "smoker" column we did not read. Smokers usually pay much more for health insurance.
So these are probably real customers (smokers), not errors.
My decision: keep them, and check the smoker column before deciding. If we delete them, the model will not learn the cost of smokers.

## Task4

In [5]:
def min_max(col):
    return (col - col.min()) / (col.max() - col.min())

age_n = min_max(age)
bmi_n = min_max(bmi)
charges_n = min_max(charges)
for name, col in [("age", age_n), ("bmi", bmi_n), ("charges", charges_n)]:
    print(name, "min:", col.min(), "max:", col.max())

print("Client 1 before:", age[0], bmi[0], charges[0])
print("Client 1 after:", round(age_n[0], 3), round(bmi_n[0], 3), round(charges_n[0], 3))

charges_z = (charges - charges.mean()) / charges.std()
print("z mean:", round(charges_z.mean(), 4), "| z std:", round(charges_z.std(), 4))

# new client: age 40, charges 20,000 (using the ORIGINAL min and max)
new_age = (40 - age.min()) / (age.max() - age.min())
new_charges = (20000 - charges.min()) / (charges.max() - charges.min())
print("New client:", round(new_age, 3), round(new_charges, 3))

print("Charges range / age range:", round((charges.max() - charges.min()) / (age.max() - age.min()), 1))

age min: 0.0 max: 1.0
bmi min: 0.0 max: 1.0
charges min: 0.0 max: 1.0
Client 1 before: 19.0 27.9 16884.924
Client 1 after: 0.022 0.321 0.252
z mean: -0.0 | z std: 1.0
New client: 0.478 0.301
Charges range / age range: 1361.9


### Results
After Min-Max, all columns go from 0 to 1.
Client 1: age 19 -> 0.022, bmi 27.9 -> 0.321, charges 16,884 -> 0.252.
After z-score, charges have mean = 0 and std = 1.

### Why not calculate a new min and max for the new client?
The model learned with the original scale. If we use a new min and max, the same number would mean something different, and the model would read it wrong.
So the new client (age 40, charges 20,000) becomes 0.478 and 0.301 using the original min and max.

### Decision
The range of charges is about 1,362 times the range of age.
Without normalization, the model would think charges are the most important column, only because its numbers are much bigger.
This is wrong, because a big number does not mean a more important column. Normalization puts all columns on the same scale so the model treats them fairly.

## Task5

In [6]:
rng = np.random.default_rng(7)
brightness = 1000 + rng.normal(0, 1, 240)
for start in [30, 102, 174]:          # hidden from the astronomers!
    brightness[start:start + 5] -= 20

print("Mean:", round(brightness.mean(), 2), "| Median:", round(np.median(brightness), 2))

z = (brightness - brightness.mean()) / brightness.std()
dips = z < -3
hours = np.arange(240)[dips]
print("Readings with z < -3:", dips.sum())
print("Hours:", hours)

# start of each transit: the hour before it was not low
gaps = np.diff(hours)
starts = hours[np.r_[True, gaps > 1]]
print("Transit starts:", starts)

period = np.diff(starts)
print("Time between transits:", period, "hours =", period / 24, "days")

normal = brightness[~dips].mean()
low = brightness[dips].mean()
print("Dip:", round(normal - low, 2), "| percent:", round((normal - low) / normal * 100, 2), "%")

Mean: 998.6 | Median: 999.84
Readings with z < -3: 15
Hours: [ 30  31  32  33  34 102 103 104 105 106 174 175 176 177 178]
Transit starts: [ 30 102 174]
Time between transits: [72 72] hours = [3. 3.] days
Dip: 20.04 | percent: 2.0 %


### Results
Mean: 998.6, Median: 999.84.
15 readings have z < -3, at hours 30-34, 102-106, and 174-178.
Each transit starts at hour 30, 102, and 174, and lasts 5 hours.
The time between transits is 72 hours = 3 days. This is the orbit period of the planet.
The brightness drops by about 20 units, which is 2.0%.

### Thinking
If we deleted all outliers here, we would delete the planet itself, the most important discovery in the data.
Lesson: before deleting any outlier, I must ask "is it an error, or is it real information?" and check the context first.